<a href="https://colab.research.google.com/github/Trondster/CS4140//blob/main/train/drone-classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install "tensorflow==2.19.0" tf-keras tensorflow-model-optimization ai-edge-litert -q

In [2]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import numpy as np
import matplotlib.pyplot as plt
import random

import tensorflow as tf
import tf_keras as keras
from tf_keras import layers
import tensorflow_model_optimization as tfmot
from ai_edge_litert import interpreter as litert

print("TensorFlow:", tf.__version__)
print("Keras (legacy):", keras.__version__)

TensorFlow: 2.19.0
Keras (legacy): 2.19.0


In [15]:
InputWidth = 53
InputHeight = 40

def create_model():
    # Define two input layers
    image1_input = layers.Input(shape=(InputWidth, InputHeight, 1), name='image1_input')
    image2_input = layers.Input(shape=(InputWidth, InputHeight, 1), name='image2_input')

    # Flatten each image input
    flat_image1 = layers.Flatten()(image1_input)
    flat_image2 = layers.Flatten()(image2_input)

    # Concatenate the two flattened tensors
    concatenated = layers.concatenate([flat_image1, flat_image2], axis=-1) # Concatenate along the feature axis

    # Define the dense layers
    x = layers.Dense(40, activation="relu")(concatenated)
    x = layers.Dense(16, activation="relu")(x)
    x = layers.Dense(8, activation="relu")(x)
    output = layers.Dense(1, activation="softmax")(x) # This is the output layer of the model

    # Create the Keras Model using the Functional API
    model = keras.Model(inputs=[image1_input, image2_input], outputs=output)

    model.compile(
        loss="sparse_categorical_crossentropy",
        optimizer="adam",
        metrics=["accuracy"],
    )
    return model

model = create_model()
model.summary()

Model: "model_10"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 image1_input (InputLayer)   [(None, 53, 40, 1)]          0         []                            
                                                                                                  
 image2_input (InputLayer)   [(None, 53, 40, 1)]          0         []                            
                                                                                                  
 flatten_18 (Flatten)        (None, 2120)                 0         ['image1_input[0][0]']        
                                                                                                  
 flatten_19 (Flatten)        (None, 2120)                 0         ['image2_input[0][0]']        
                                                                                           

# Prepare Data

Normalize pixel values to `[0, 1]`. We keep images in their original shape — the `Flatten` layer inside the model handles the flattening automatically.

In [ ]:
# Normalize to float32 in [0, 1] — keep the 2-D shape for the Flatten layer
x_train = x_train.astype("float32") / 255.0
x_test  = x_test.astype("float32")  / 255.0

print("x_train shape:", x_train.shape)
print("x_test  shape:", x_test.shape)

# Train and Save Model

In [ ]:
model.fit(
    x=x_train,
    y=y_train,
    batch_size=128,
    epochs=10,
    validation_split=0.1,
    verbose=1,
)

scores = model.evaluate(x_test, y_test, verbose=2)
print("Test Loss:    ", scores[0])
print("Test Accuracy:", scores[1])

model.save("mnist_model.keras")

# Full INT8 Post-Training Quantization (PTQ)

In [ ]:
# Use 100 samples from the training set as the representative dataset
x_rep = x_train[:100].reshape(-1, InputWidth, InputHeight)   # keep 2-D shape

def representative_dataset():
    for sample in x_rep:
        yield [sample[np.newaxis, :, :]]     # shape (1, InputWidth, InputHeight)

converter_ptq = tf.lite.TFLiteConverter.from_keras_model(model)
converter_ptq.optimizations = [tf.lite.Optimize.DEFAULT]
converter_ptq.representative_dataset = representative_dataset
converter_ptq.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter_ptq.inference_input_type  = tf.int8
converter_ptq.inference_output_type = tf.int8

tflite_ptq = converter_ptq.convert()
open("model_ptq_int8.tflite", "wb").write(tflite_ptq)
print("PTQ INT8 model saved — size:", len(tflite_ptq), "bytes")

# Quantization-Aware Training (QAT)

In [ ]:
# 1. Wrap the trained model with QAT fake-quantization
# Note: tfmot.quantization.keras.quantize_model() requires a Functional model.
functional_model = keras.Model(inputs=model.input, outputs=model.output)
q_aware_model = tfmot.quantization.keras.quantize_model(functional_model)

q_aware_model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"],
)
q_aware_model.summary()

In [ ]:
# 2. Fine-tune for 5 epochs
q_aware_model.fit(
    x=x_train,
    y=y_train,
    batch_size=128,
    epochs=5,
    validation_split=0.1,
    verbose=1,
)

In [ ]:
# 3. Convert the QAT model to INT8 TFLite
converter_qat = tf.lite.TFLiteConverter.from_keras_model(q_aware_model)
converter_qat.optimizations = [tf.lite.Optimize.DEFAULT]
converter_qat.representative_dataset = representative_dataset
converter_qat.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter_qat.inference_input_type  = tf.int8
converter_qat.inference_output_type = tf.int8

tflite_qat = converter_qat.convert()
open("model_qat_int8.tflite", "wb").write(tflite_qat)
print("QAT INT8 model saved — size:", len(tflite_qat), "bytes")

# Model Size Comparison

INT8 models store weights as 8-bit integers instead of 32-bit floats — roughly **4× smaller**. On a microcontroller with only tens of kilobytes of flash, this difference matters enormously.

In [ ]:
# Save the float model as TFLite for a fair comparison
converter_float = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_float = converter_float.convert()
open("model_float32.tflite", "wb").write(tflite_float)

sizes = {
    "Float32 TFLite": os.path.getsize("model_float32.tflite"),
    "PTQ INT8      ": os.path.getsize("model_ptq_int8.tflite"),
    "QAT INT8      ": os.path.getsize("model_qat_int8.tflite"),
}

print(f"{'Model':<20} {'Size (bytes)':>14} {'Size (KB)':>10}")
print("-" * 46)
for name, size in sizes.items():
    print(f"{name:<20} {size:>14,} {size/1024:>9.1f}")

# Test TFLite Models

Test **sample-by-sample** — exactly how inference runs on a microcontroller. For INT8 models the input must be scaled from `float32` to `int8` using the quantization parameters reported by the interpreter.

In [ ]:
def evaluate_tflite(model_path, x, y_true, max_samples=1000):
    """Run TFLite inference sample-by-sample and return accuracy."""
    interpreter = litert.Interpreter(model_path=model_path)
    interpreter.allocate_tensors()

    input_details  = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    input_dtype = input_details[0]["dtype"]
    is_int8 = (input_dtype == np.int8)

    if is_int8:
        scale, zero_point = input_details[0]["quantization"]

    correct = 0
    for i in range(min(max_samples, len(x))):
        sample = x[i:i+1]                       # shape (1, 28, 28)

        if is_int8:
            sample = (sample / scale + zero_point).astype(np.int8)

        interpreter.set_tensor(input_details[0]["index"], sample)
        interpreter.invoke()
        output = interpreter.get_tensor(output_details[0]["index"])
        pred = np.argmax(output[0])
        if pred == y_true[i]:
            correct += 1

    return correct / min(max_samples, len(x))

acc_ptq = evaluate_tflite("model_ptq_int8.tflite", x_test, y_test)
acc_qat = evaluate_tflite("model_qat_int8.tflite", x_test, y_test)

print(f"PTQ INT8 accuracy : {acc_ptq:.4f}")
print(f"QAT INT8 accuracy : {acc_qat:.4f}")

# Export as C Arrays

The `.tflite` binary is embedded in a C header as a `const` byte array. The MCU firmware reads this array directly from flash.

Good practices for MCU C headers:
- `#ifndef` / `#define` / `#endif` — prevent double-inclusion
- `alignas(16)` — ensures 16-byte alignment for SIMD-like hardware (e.g. DSP cores)
- `const` — allows the linker to place the array in read-only flash

In [ ]:
import binascii
import datetime

def to_c_array(data: bytes) -> str:
    """Convert bytes to a comma-separated C hex array, 12 bytes per line."""
    hexstr = binascii.hexlify(data).decode("ascii").upper()
    values = ["0x" + hexstr[i:i+2] for i in range(0, len(hexstr), 2)]
    lines  = [", ".join(values[i:i+12]) for i in range(0, len(values), 12)]
    return ",\n  ".join(lines)

def write_c_header(tflite_path: str, header_path: str, array_name: str, quant_type: str):
    data = open(tflite_path, "rb").read()
    guard = array_name.upper() + "_H"
    body = (
        f"// Auto-generated on {datetime.date.today()}\n"
        f"// Model: {tflite_path}  |  Quantization: {quant_type}  |  Size: {len(data)} bytes\n"
        f"#ifndef {guard}\n"
        f"#define {guard}\n"
        f"#include <stdint.h>\n"
        f"#include <stdalign.h>\n\n"
        f"alignas(16) const uint8_t {array_name}[] = {{\n  "
        + to_c_array(data) +
        f"\n}};\n"
        f"const unsigned int {array_name}_len = {len(data)};\n\n"
        f"#endif  // {guard}\n"
    )
    open(header_path, "w").write(body)
    print(f"Written {header_path}  ({len(data)} bytes)")

write_c_header("model_ptq_int8.tflite", "model_ptq_int8.h", "tf_model_ptq", "INT8 PTQ")
write_c_header("model_qat_int8.tflite", "model_qat_int8.h", "tf_model_qat", "INT8 QAT")